# T05 对冲策略重构项目

## 项目概述

本项目专注于债券或ETF的对冲策略分析，包括：
- **市场环境分析**：波动率、相关性、趋势环境分析
- **历史场景识别**：识别历史上的类似市场环境
- **策略回测**：不同对冲比率的收益和风险分析
- **风险评估**：VaR、CVaR、最大回撤等风险指标计算
- **报告生成**：生成交互式HTML报告

### 数据源
- MySQL数据库（marketinfo_fund表）
- Wind API（数据更新）

### 输出结果
- 策略净值对比图表
- 市场环境热力图
- 策略信号强度图
- HTML分析报告

---
## 1. 环境配置

In [ ]:
# 导入标准库
import os
import sys
import json
import warnings
from datetime import datetime, date, timedelta
from time import sleep
import time
import random
import re

warnings.filterwarnings('ignore')

In [ ]:
# 导入第三方库
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
import pymysql
import requests
from bs4 import BeautifulSoup
import chardet

# 可视化库
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # 非交互式后端
import seaborn as sns

# 统计分析
from scipy import stats

# 交互式可视化
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.subplots import make_subplots

# 模板引擎
from jinja2 import Template

# Wind API（需要安装Wind终端）
try:
    from WindPy import w
    WIND_AVAILABLE = True
except ImportError:
    WIND_AVAILABLE = False
    print("警告: WindPy未安装，数据更新功能不可用")

# 浏览器控制
import webbrowser

In [ ]:
# 导入项目配置
from config import (
    DB_CONFIG,
    ANALYSIS_CONFIG,
    VIZ_CONFIG,
    get_database_url
)

print("环境配置完成")

In [ ]:
# 配置中文字体
def setup_chinese_font():
    """配置matplotlib中文字体"""
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    print("中文字体配置完成")

setup_chinese_font()

---
## 2. 数据获取

In [ ]:
class DatabaseManager:
    """数据库管理类"""
    
    def __init__(self):
        """初始化数据库连接"""
        self.engine = None
        self.connection = None
        
    def connect(self):
        """建立数据库连接"""
        try:
            db_url = get_database_url()
            self.engine = sqlalchemy.create_engine(
                db_url,
                poolclass=sqlalchemy.pool.NullPool
            )
            self.connection = self.engine.connect()
            print("数据库连接成功")
            return True
        except Exception as e:
            print(f"数据库连接失败: {str(e)}")
            return False
    
    def disconnect(self):
        """断开数据库连接"""
        if self.connection:
            self.connection.close()
        if self.engine:
            self.engine.dispose()
        print("数据库连接已关闭")
    
    def execute_query(self, sql):
        """执行SQL查询并返回DataFrame"""
        try:
            return pd.read_sql(sql, self.engine)
        except Exception as e:
            print(f"查询执行失败: {str(e)}")
            return pd.DataFrame()
    
    def get_latest_date(self, table_name, trade_code):
        """获取表中某标的的最新数据日期"""
        sql = f"""
        SELECT MAX(dt) as latest_date 
        FROM {table_name} 
        WHERE trade_code = '{trade_code}'
        """
        result = self.execute_query(sql)
        if not result.empty and result['latest_date'].iloc[0]:
            return result['latest_date'].iloc[0]
        return None

In [ ]:
# 创建数据库连接
db = DatabaseManager()
db.connect()

In [ ]:
class DataFetcher:
    """数据获取类"""
    
    def __init__(self, db_manager):
        self.db = db_manager
        self.table_name = 'marketinfo_fund'
    
    def get_market_data(self, trade_code, start_date='2023-01-14'):
        """获取指定标的的市场数据
        
        参数:
            trade_code: 标的代码
            start_date: 起始日期
            
        返回:
            DataFrame: 包含dt, open, high, low, close, volume, vwap列
        """
        sql = f"""
        SELECT dt, open, high, low, close, volume, vwap
        FROM {self.table_name} 
        WHERE trade_code = '{trade_code}'
        AND dt >= '{start_date}'
        ORDER BY dt
        """
        return self.db.execute_query(sql)
    
    def get_paired_data(self, trade_code1, trade_code2, start_date='2023-01-14'):
        """获取两个标的的配对数据
        
        参数:
            trade_code1: 标的1代码
            trade_code2: 标的2代码
            start_date: 起始日期
            
        返回:
            DataFrame: 合并后的数据
        """
        # 获取标的1数据
        df1 = self.get_market_data(trade_code1, start_date)
        print(f"标的1 ({trade_code1}) 数据条数: {len(df1)}")
        
        # 获取标的2数据
        df2 = self.get_market_data(trade_code2, start_date)
        print(f"标的2 ({trade_code2}) 数据条数: {len(df2)}")
        
        if df1.empty or df2.empty:
            print("警告: 数据为空")
            return pd.DataFrame()
        
        # 合并数据
        merged_df = pd.merge(
            df1, df2, 
            on='dt', 
            how='inner',
            suffixes=('_1', '_2')
        )
        print(f"合并后数据条数: {len(merged_df)}")
        
        return merged_df

In [ ]:
# 测试数据获取
fetcher = DataFetcher(db)

# 示例标的配置
TRADE_CODE_1 = '511220.SH'  # 城投ETF
TRADE_CODE_2 = '518880.SH'  # 黄金ETF
NAME_1 = '城投ETF'
NAME_2 = '黄金ETF'

raw_data = fetcher.get_paired_data(TRADE_CODE_1, TRADE_CODE_2)
raw_data.head()

---
## 3. 数据处理

In [ ]:
class DataProcessor:
    """数据处理类"""
    
    def __init__(self, df, name1='标的1', name2='标的2'):
        """
        参数:
            df: 原始数据DataFrame
            name1: 标的1名称
            name2: 标的2名称
        """
        self.raw_df = df.copy()
        self.name1 = name1
        self.name2 = name2
        self.df = None
    
    def process(self):
        """执行完整的数据处理流程"""
        if self.raw_df.empty:
            print("错误: 原始数据为空")
            return False
        
        # 1. 日期处理
        self._process_dates()
        
        # 2. 计算收益率
        self._calculate_returns()
        
        # 3. 处理异常值
        self._handle_outliers()
        
        print(f"数据处理完成，最终数据条数: {len(self.df)}")
        return True
    
    def _process_dates(self):
        """处理日期列"""
        self.df = self.raw_df.copy()
        self.df['dt'] = pd.to_datetime(self.df['dt'])
        self.df = self.df.set_index('dt')
        print(f"日期范围: {self.df.index.min()} 至 {self.df.index.max()}")
    
    def _calculate_returns(self):
        """计算收益率"""
        self.df['return_1'] = self.df['close_1'].pct_change()
        self.df['return_2'] = self.df['close_2'].pct_change()
    
    def _handle_outliers(self):
        """处理异常值（单日涨跌幅超过20%）"""
        max_return = ANALYSIS_CONFIG['outlier']['max_return']
        
        # 保留最后一天数据
        last_day = self.df.iloc[[-1]]
        temp_df = self.df.iloc[:-1]
        
        # 识别异常值
        mask = (abs(temp_df['return_1']) < max_return) & (abs(temp_df['return_2']) < max_return)
        removed_count = (~mask).sum()
        
        if removed_count > 0:
            print(f"移除 {removed_count} 个异常值日期")
        
        # 合并数据
        self.df = pd.concat([temp_df[mask], last_day])
    
    def get_processed_data(self):
        """获取处理后的数据"""
        return self.df

In [ ]:
# 数据处理
processor = DataProcessor(raw_data, NAME_1, NAME_2)
processor.process()

df = processor.get_processed_data()
df.tail()

In [ ]:
# 数据概览
print("="*50)
print("数据概览")
print("="*50)
print(f"数据条数: {len(df)}")
print(f"日期范围: {df.index[0].strftime('%Y-%m-%d')} 至 {df.index[-1].strftime('%Y-%m-%d')}")
print(f"\n{NAME_1} 收盘价范围: {df['close_1'].min():.2f} - {df['close_1'].max():.2f}")
print(f"{NAME_2} 收盘价范围: {df['close_2'].min():.2f} - {df['close_2'].max():.2f}")
print(f"\n{NAME_1} 年化波动率: {df['return_1'].std() * np.sqrt(252):.2%}")
print(f"{NAME_2} 年化波动率: {df['return_2'].std() * np.sqrt(252):.2%}")

---
## 4. 核心逻辑

In [ ]:
class MarketAnalyzer:
    """市场分析器"""
    
    def __init__(self, df, name1='标的1', name2='标的2'):
        """
        参数:
            df: 处理后的数据DataFrame
            name1: 标的1名称
            name2: 标的2名称
        """
        self.df = df
        self.name1 = name1
        self.name2 = name2
    
    def analyze_market_conditions(self):
        """分析当前市场环境
        
        返回:
            dict: 包含波动率环境、相关性环境、趋势环境
        """
        try:
            # 1. 波动率环境
            vol_regime = {
                f'{self.name1}波动率': self.df['return_1'].std() * np.sqrt(252),
                f'{self.name2}波动率': self.df['return_2'].std() * np.sqrt(252),
                '波动率比值': self.df['return_1'].std() / self.df['return_2'].std(),
                '波动率百分位': {
                    self.name1: stats.percentileofscore(
                        self.df['return_1'].rolling(20).std().dropna(), 
                        self.df['return_1'].std()
                    ),
                    self.name2: stats.percentileofscore(
                        self.df['return_2'].rolling(20).std().dropna(), 
                        self.df['return_2'].std()
                    )
                }
            }
            
            # 2. 相关性环境
            correlation = self.df['return_1'].rolling(20).corr(self.df['return_2'])
            corr_regime = {
                '当前相关性': correlation.iloc[-1],
                '相关性稳定性': correlation.std(),
                '相关性趋势': correlation.iloc[-1] - correlation.iloc[-20] if len(correlation) >= 20 else 0
            }
            
            # 3. 趋势环境
            trend_regime = {
                f'{self.name1}RSI': self._calculate_rsi(self.df['return_1']).iloc[-1],
                f'{self.name2}RSI': self._calculate_rsi(self.df['return_2']).iloc[-1],
                f'{self.name1}动能': self.df['return_1'].mean() / self.df['return_1'].std() if self.df['return_1'].std() != 0 else 0,
                f'{self.name2}动能': self.df['return_2'].mean() / self.df['return_2'].std() if self.df['return_2'].std() != 0 else 0
            }
            
            return {
                '波动率环境': vol_regime,
                '相关性环境': corr_regime,
                '趋势环境': trend_regime
            }
            
        except Exception as e:
            print(f"分析市场环境失败: {str(e)}")
            return None
    
    def _calculate_rsi(self, returns, periods=14):
        """计算RSI指标"""
        delta = returns
        gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
        rs = gain / loss
        return 100 - (100 / (1 + rs))

In [ ]:
    def get_historical_scenarios(self):
        """识别历史上的类似场景
        
        返回:
            DataFrame: 包含历史场景分析结果
        """
        try:
            # 计算滚动指标
            rolling_vol_1 = self.df['return_1'].rolling(20).std() * np.sqrt(252)
            rolling_vol_2 = self.df['return_2'].rolling(20).std() * np.sqrt(252)
            rolling_corr = self.df['return_1'].rolling(20).corr(self.df['return_2'])
            
            # 当前环境指标
            current_vol_1 = rolling_vol_1.iloc[-1]
            current_vol_2 = rolling_vol_2.iloc[-1]
            current_corr = rolling_corr.iloc[-1]
            
            # 定义相似性条件
            vol_similar = (
                (rolling_vol_1 > current_vol_1 * 0.8) &
                (rolling_vol_1 < current_vol_1 * 1.2) &
                (rolling_vol_2 > current_vol_2 * 0.8) &
                (rolling_vol_2 < current_vol_2 * 1.2)
            )
            
            corr_similar = (
                (rolling_corr > current_corr - 0.2) &
                (rolling_corr < current_corr + 0.2)
            )
            
            # 找到类似场景
            similar_days = self.df[vol_similar & corr_similar]
            
            # 分析这些场景下的表现
            scenarios = []
            for i in range(len(similar_days) - 20):
                scenario_start = similar_days.index[i]
                forward_20d = self.df.loc[scenario_start:].head(20)
                
                if len(forward_20d) == 20:
                    scenario = {
                        '日期': scenario_start,
                        f'{self.name1}20日收益': forward_20d['close_1'].iloc[-1] / forward_20d['close_1'].iloc[0] - 1,
                        f'{self.name2}20日收益': forward_20d['close_2'].iloc[-1] / forward_20d['close_2'].iloc[0] - 1,
                        '相关性': rolling_corr.loc[scenario_start:scenario_start + timedelta(days=20)].mean(),
                        '波动率比值': (
                            rolling_vol_1.loc[scenario_start:scenario_start + timedelta(days=20)] /
                            rolling_vol_2.loc[scenario_start:scenario_start + timedelta(days=20)]
                        ).mean()
                    }
                    scenarios.append(scenario)
            
            return pd.DataFrame(scenarios)
            
        except Exception as e:
            print(f"分析历史场景失败: {str(e)}")
            return pd.DataFrame()

In [ ]:
    def calculate_optimal_weights(self):
        """计算最优配置权重
        
        返回:
            dict: 包含风险平价权重和动态调整权重
        """
        # 计算风险平价权重
        vol_1 = self.df['return_1'].rolling(20).std()
        vol_2 = self.df['return_2'].rolling(20).std()
        w_rp_1 = 1/vol_1 / (1/vol_1 + 1/vol_2)
        w_rp_2 = 1 - w_rp_1
        
        # 计算动态调整权重
        corr = self.df['return_1'].rolling(20).corr(self.df['return_2'])
        momentum_1 = self.df['return_1'].rolling(60).mean()
        momentum_2 = self.df['return_2'].rolling(60).mean()
        
        # 基础权重
        base_weight = 0.6
        
        # 相关性调整
        corr_adj = -0.2 * corr
        
        # 动量调整
        mom_adj = 0.1 * (momentum_1 > momentum_2).astype(float)
        
        # 波动率调整
        vol_adj = -0.1 * (vol_1 > vol_2).astype(float)
        
        # 最终动态权重
        w_dyn_1 = (base_weight + corr_adj + mom_adj + vol_adj).clip(0.3, 0.8)
        w_dyn_2 = 1 - w_dyn_1
        
        return {
            '风险平价': {
                f'{self.name1}权重': w_rp_1.iloc[-1],
                f'{self.name2}权重': w_rp_2.iloc[-1],
                '计算依据': f'基于20日波动率（{self.name1}: {vol_1.iloc[-1]:.2%}, {self.name2}: {vol_2.iloc[-1]:.2%}）'
            },
            '动态调整': {
                f'{self.name1}权重': w_dyn_1.iloc[-1],
                f'{self.name2}权重': w_dyn_2.iloc[-1],
                '计算依据': f'相关性:{corr.iloc[-1]:.2f}, 动量比:{(momentum_1/momentum_2).iloc[-1]:.2f}'
            }
        }

In [ ]:
    def calculate_strategy_nav(self, weights):
        """计算策略净值
        
        参数:
            weights: 权重配置，可以是元组(固定权重)、None(风险平价)、'dynamic'(动态调整)
            
        返回:
            Series: 策略净值序列
        """
        try:
            if weights is None:
                # 风险平价权重
                vol_1 = self.df['return_1'].rolling(20).std()
                vol_2 = self.df['return_2'].rolling(20).std()
                w_1 = 1/vol_1 / (1/vol_1 + 1/vol_2)
                w_1 = w_1.fillna(0.5)
                w_1_aligned = w_1.shift(1)
                w_1_aligned.iloc[0] = 0.5
                
                returns = self.df['return_1'] * w_1_aligned + self.df['return_2'] * (1 - w_1_aligned)
                
            elif isinstance(weights, str) and weights == 'dynamic':
                # 动态调整权重
                base_weight = 0.6
                corr = self.df['return_1'].rolling(20).corr(self.df['return_2'])
                corr_adj = -0.2 * corr
                momentum_1 = self.df['return_1'].rolling(60).mean()
                momentum_2 = self.df['return_2'].rolling(60).mean()
                mom_adj = 0.1 * (momentum_1 > momentum_2).astype(float)
                vol_1 = self.df['return_1'].rolling(20).std()
                vol_2 = self.df['return_2'].rolling(20).std()
                vol_adj = -0.1 * (vol_1 > vol_2).astype(float)
                
                w_1 = (base_weight + corr_adj + mom_adj + vol_adj).clip(0.3, 0.8)
                w_1 = w_1.fillna(0.5)
                w_1_aligned = w_1.shift(1)
                w_1_aligned.iloc[0] = 0.5
                
                returns = self.df['return_1'] * w_1_aligned + self.df['return_2'] * (1 - w_1_aligned)
                
            elif isinstance(weights, tuple) and len(weights) == 2:
                # 固定权重
                w_1, w_2 = weights
                returns = self.df['return_1'] * w_1 + self.df['return_2'] * w_2
            else:
                raise ValueError("无效的权重参数")
            
            # 计算净值
            nav = (1 + returns).cumprod()
            return nav
            
        except Exception as e:
            print(f"计算策略净值失败: {str(e)}")
            return pd.Series(index=self.df.index)

In [ ]:
    def calculate_strategy_metrics(self):
        """计算各策略的表现指标
        
        返回:
            dict: 各策略的指标字典
        """
        strategies = {
            f'{self.name1}': (1.0, 0.0),
            f'{self.name2}': (0.0, 1.0),
            '固定配置(70/30)': (0.7, 0.3),
            '固定配置(50/50)': (0.5, 0.5),
            '风险平价': None,
            '动态调整': 'dynamic'
        }
        
        metrics = {}
        risk_free_rate = 0.02
        
        for name, weights in strategies.items():
            nav = self.calculate_strategy_nav(weights)
            returns = nav.pct_change().dropna()
            
            annual_return = (nav.iloc[-1] ** (252/len(nav)) - 1)
            annual_vol = returns.std() * np.sqrt(252)
            sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol != 0 else 0
            max_drawdown = (nav / nav.expanding().max() - 1).min()
            win_rate = (returns > 0).mean()
            
            metrics[name] = {
                '年化收益率': annual_return,
                '年化波动率': annual_vol,
                '夏普比率': sharpe,
                '最大回撤': max_drawdown,
                '胜率': win_rate
            }
        
        return metrics

    def generate_allocation_advice(self, shares_1, shares_2, cash):
        """生成配置建议
        
        参数:
            shares_1: 标的1持仓数量
            shares_2: 标的2持仓数量
            cash: 可用现金
            
        返回:
            list: 配置建议列表
        """
        # 获取最新价格
        price_1 = self.df['close_1'].iloc[-1]
        price_2 = self.df['close_2'].iloc[-1]
        
        # 计算当前持仓市值
        value_1 = shares_1 * price_1
        value_2 = shares_2 * price_2
        total_value = value_1 + value_2 + cash
        
        # 获取最优权重
        weights = self.calculate_optimal_weights()
        
        advice = []
        for strategy, w in weights.items():
            target_1 = total_value * w[f'{self.name1}权重']
            target_2 = total_value * w[f'{self.name2}权重']
            
            diff_1 = target_1 - value_1
            diff_2 = target_2 - value_2
            
            target_config = {
                f'{self.name1}目标持仓': f"{int(target_1 / price_1)}份",
                f'{self.name2}目标持仓': f"{int(target_2 / price_2)}份",
                f'{self.name1}目标占比': f"{w[f'{self.name1}权重']:.1%}",
                f'{self.name2}目标占比': f"{w[f'{self.name2}权重']:.1%}",
                '建议操作': []
            }
            
            if abs(diff_1) > total_value * 0.05:
                action = "增配" if diff_1 > 0 else "减配"
                target_config['建议操作'].append(f"{action}{self.name1} {abs(int(diff_1 / price_1))}份")
            
            if abs(diff_2) > total_value * 0.05:
                action = "增配" if diff_2 > 0 else "减配"
                target_config['建议操作'].append(f"{action}{self.name2} {abs(int(diff_2 / price_2))}份")
            
            advice.append({
                '策略': strategy,
                '目标配置': target_config,
                '计算依据': w['计算依据']
            })
        
        return advice

In [ ]:
# 创建分析器并执行分析
analyzer = MarketAnalyzer(df, NAME_1, NAME_2)

# 市场环境分析
market_conditions = analyzer.analyze_market_conditions()
print("\n=== 市场环境分析 ===")
for key, value in market_conditions.items():
    print(f"\n{key}:")
    if isinstance(value, dict):
        for k, v in value.items():
            if isinstance(v, dict):
                print(f"  {k}:")
                for kk, vv in v.items():
                    print(f"    {kk}: {vv:.2f}")
            else:
                print(f"  {k}: {v:.4f}")

In [ ]:
# 策略指标分析
strategy_metrics = analyzer.calculate_strategy_metrics()

print("\n=== 策略表现指标 ===")
metrics_df = pd.DataFrame(strategy_metrics).T
metrics_df

In [ ]:
# 最优权重计算
optimal_weights = analyzer.calculate_optimal_weights()

print("\n=== 最优配置权重 ===")
for strategy, weights in optimal_weights.items():
    print(f"\n{strategy}:")
    for key, value in weights.items():
        print(f"  {key}: {value}")

---
## 5. 执行与测试

In [ ]:
# 配置测试参数
TEST_SHARES_1 = 10000  # 标的1持仓数量
TEST_SHARES_2 = 5000   # 标的2持仓数量
TEST_CASH = 100000     # 可用现金

# 生成配置建议
advice = analyzer.generate_allocation_advice(TEST_SHARES_1, TEST_SHARES_2, TEST_CASH)

print("\n=== 配置建议 ===")
for item in advice:
    print(f"\n策略: {item['策略']}")
    print(f"计算依据: {item['计算依据']}")
    print("目标配置:")
    for key, value in item['目标配置'].items():
        if key != '建议操作':
            print(f"  {key}: {value}")
    if item['目标配置']['建议操作']:
        print(f"建议操作: {', '.join(item['目标配置']['建议操作'])}")
    else:
        print("建议操作: 当前配置合理，无需调整")

In [ ]:
# 绘制策略净值对比图
def plot_strategy_nav_comparison(analyzer):
    """绘制策略净值对比图"""
    fig, ax = plt.subplots(figsize=(14, 8))
    
    strategies = {
        f'{analyzer.name1}': (1.0, 0.0),
        f'{analyzer.name2}': (0.0, 1.0),
        '固定配置(70/30)': (0.7, 0.3),
        '固定配置(50/50)': (0.5, 0.5),
        '风险平价': None,
        '动态调整': 'dynamic'
    }
    
    for name, weights in strategies.items():
        nav = analyzer.calculate_strategy_nav(weights)
        ax.plot(nav.index, nav.values, label=name, alpha=0.8, linewidth=2)
    
    ax.set_title('策略净值对比', fontsize=14)
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('净值', fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('output/strategy_nav_comparison.png', dpi=300)
    plt.show()
    print("图表已保存至 output/strategy_nav_comparison.png")

plot_strategy_nav_comparison(analyzer)

In [ ]:
# 生成交互式Plotly图表
def create_plotly_nav_chart(analyzer):
    """创建交互式净值对比图"""
    fig = go.Figure()
    
    strategies = {
        f'{analyzer.name1}': (1.0, 0.0),
        f'{analyzer.name2}': (0.0, 1.0),
        '固定配置(70/30)': (0.7, 0.3),
        '固定配置(50/50)': (0.5, 0.5),
        '风险平价': None,
        '动态调整': 'dynamic'
    }
    
    for name, weights in strategies.items():
        nav = analyzer.calculate_strategy_nav(weights)
        fig.add_trace(
            go.Scatter(
                x=nav.index,
                y=nav.values,
                name=name,
                mode='lines',
                hovertemplate='%{x|%Y-%m-%d}<br>净值: %{y:.2f}<extra></extra>'
            )
        )
    
    fig.update_layout(
        title='策略净值对比 (2023年至今)',
        xaxis_title='日期',
        yaxis_title='净值',
        height=600,
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02
        )
    )
    
    return fig

nav_fig = create_plotly_nav_chart(analyzer)
nav_fig.show()

In [ ]:
# 保存图表
nav_fig.write_html('output/strategy_nav_comparison.html')
print("交互式图表已保存至 output/strategy_nav_comparison.html")

---
## 6. 报告生成

In [ ]:
class ReportGenerator:
    """报告生成器"""
    
    def __init__(self, analyzer, output_dir='output'):
        self.analyzer = analyzer
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
    
    def create_nav_figure(self):
        """创建策略净值对比图"""
        fig = go.Figure()
        
        strategies = {
            f'{self.analyzer.name1}': (1.0, 0.0),
            f'{self.analyzer.name2}': (0.0, 1.0),
            '固定配置(70/30)': (0.7, 0.3),
            '固定配置(50/50)': (0.5, 0.5),
            '风险平价': None,
            '动态调整': 'dynamic'
        }
        
        for name, weights in strategies.items():
            nav = self.analyzer.calculate_strategy_nav(weights)
            fig.add_trace(
                go.Scatter(
                    x=nav.index,
                    y=nav.values,
                    name=name,
                    hovertemplate='%{x|%Y-%m-%d}<br>净值: %{y:.2f}<extra></extra>'
                )
            )
        
        fig.update_layout(
            title='策略净值对比',
            height=500,
            showlegend=True,
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.05),
            hovermode='x unified',
            xaxis_title="日期",
            yaxis_title="净值"
        )
        
        return fig
    
    def create_signals_figure(self):
        """创建策略信号图"""
        fig = go.Figure()
        
        signals = {
            '趋势信号': self.analyzer.df['close_1'].pct_change(20).iloc[-1],
            '波动信号': (self.analyzer.df['return_1'].std() - self.analyzer.df['return_2'].std()) * np.sqrt(252),
            '相关信号': self.analyzer.df['return_1'].rolling(20).corr(self.analyzer.df['return_2']).iloc[-1],
            '动量信号': (self.analyzer.df['close_1'].pct_change(60) - self.analyzer.df['close_2'].pct_change(60)).iloc[-1]
        }
        
        names = list(signals.keys())
        values = list(signals.values())
        colors = ['red' if v > 0 else 'green' for v in values]
        
        fig.add_trace(
            go.Bar(
                y=names,
                x=values,
                orientation='h',
                marker_color=colors,
                text=[f'{v:.2%}' for v in values],
                textposition='outside'
            )
        )
        
        fig.add_shape(type='line', x0=0, x1=0, y0=-0.5, y1=len(names)-0.5,
                      line=dict(color='black', width=1))
        
        fig.update_layout(
            title='策略信号强度',
            height=400,
            showlegend=False
        )
        
        return fig

In [ ]:
    def generate_html_report(self, shares_1, shares_2, cash):
        """生成HTML报告
        
        参数:
            shares_1: 标的1持仓数量
            shares_2: 标的2持仓数量
            cash: 可用现金
            
        返回:
            str: 报告文件路径
        """
        # 获取最新价格
        price_1 = self.analyzer.df['close_1'].iloc[-1]
        price_2 = self.analyzer.df['close_2'].iloc[-1]
        
        # 计算持仓市值
        value_1 = shares_1 * price_1
        value_2 = shares_2 * price_2
        total_value = value_1 + value_2 + cash
        
        # 生成图表
        nav_fig = self.create_nav_figure()
        signals_fig = self.create_signals_figure()
        
        # 获取配置建议
        advice = self.analyzer.generate_allocation_advice(shares_1, shares_2, cash)
        
        # 获取策略指标
        strategy_metrics = self.analyzer.calculate_strategy_metrics()
        
        # HTML模板
        template_str = """
        <!DOCTYPE html>
        <html>
        <head>
            <meta charset="UTF-8">
            <title>{{ name1 }}与{{ name2 }}对冲策略分析报告</title>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                body { font-family: Arial, sans-serif; margin: 20px; background-color: #f5f5f5; }
                .container { max-width: 1200px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; }
                .section { margin-bottom: 30px; }
                table { width: 100%; border-collapse: collapse; margin: 20px 0; }
                th, td { padding: 12px; text-align: left; border: 1px solid #ddd; }
                th { background-color: #3498db; color: white; }
                .header { background-color: #f8f9fa; padding: 20px; margin-bottom: 30px; border-radius: 5px; }
                .timestamp { color: #666; }
                h1 { color: #2c3e50; }
                h2 { color: #34495e; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
                .metric-positive { color: #27ae60; font-weight: bold; }
                .metric-negative { color: #e74c3c; font-weight: bold; }
            </style>
        </head>
        <body>
            <div class="container">
                <div class="header">
                    <h1>{{ name1 }}与{{ name2 }}对冲策略分析报告</h1>
                    <p class="timestamp">生成时间：{{ timestamp }}</p>
                </div>

                <div class="section">
                    <h2>持仓概况</h2>
                    <table>
                        <tr>
                            <th>标的</th>
                            <th>数量</th>
                            <th>市值</th>
                            <th>占比</th>
                        </tr>
                        <tr>
                            <td>{{ name1 }}</td>
                            <td>{{ "{:,.0f}".format(shares_1) }}</td>
                            <td>{{ "{:,.0f}".format(value_1) }}元</td>
                            <td>{{ "{:.1%}".format(value_1/total_value) }}</td>
                        </tr>
                        <tr>
                            <td>{{ name2 }}</td>
                            <td>{{ "{:,.0f}".format(shares_2) }}</td>
                            <td>{{ "{:,.0f}".format(value_2) }}元</td>
                            <td>{{ "{:.1%}".format(value_2/total_value) }}</td>
                        </tr>
                        <tr>
                            <td>现金</td>
                            <td>-</td>
                            <td>{{ "{:,.0f}".format(cash) }}元</td>
                            <td>{{ "{:.1%}".format(cash/total_value) }}</td>
                        </tr>
                        <tr>
                            <td><strong>总计</strong></td>
                            <td>-</td>
                            <td><strong>{{ "{:,.0f}".format(total_value) }}元</strong></td>
                            <td><strong>100%</strong></td>
                        </tr>
                    </table>
                </div>

                <div class="section">
                    <h2>策略净值对比</h2>
                    {{ nav_chart }}
                </div>

                <div class="section">
                    <h2>策略信号</h2>
                    {{ signals_chart }}
                </div>

                <div class="section">
                    <h2>策略指标</h2>
                    <table>
                        <tr>
                            <th>策略</th>
                            <th>年化收益率</th>
                            <th>年化波动率</th>
                            <th>夏普比率</th>
                            <th>最大回撤</th>
                            <th>胜率</th>
                        </tr>
                        {% for name, metrics in strategy_metrics.items() %}
                        <tr>
                            <td>{{ name }}</td>
                            <td class="{% if metrics['年化收益率'] > 0 %}metric-positive{% else %}metric-negative{% endif %}">
                                {{ "{:,.2%}".format(metrics['年化收益率']) }}
                            </td>
                            <td>{{ "{:,.2%}".format(metrics['年化波动率']) }}</td>
                            <td class="{% if metrics['夏普比率'] > 1 %}metric-positive{% elif metrics['夏普比率'] < 0 %}metric-negative{% endif %}">
                                {{ "{:,.2f}".format(metrics['夏普比率']) }}
                            </td>
                            <td class="metric-negative">{{ "{:,.2%}".format(metrics['最大回撤']) }}</td>
                            <td>{{ "{:,.2%}".format(metrics['胜率']) }}</td>
                        </tr>
                        {% endfor %}
                    </table>
                </div>

                <div class="section">
                    <h2>配置建议</h2>
                    <table>
                        <tr>
                            <th>策略</th>
                            <th>目标配置</th>
                            <th>建议操作</th>
                            <th>计算依据</th>
                        </tr>
                        {% for item in advice %}
                        <tr>
                            <td>{{ item['策略'] }}</td>
                            <td>
                                {{ name1 }}: {{ item['目标配置'][name1 + '目标持仓'] }} ({{ item['目标配置'][name1 + '目标占比'] }})<br>
                                {{ name2 }}: {{ item['目标配置'][name2 + '目标持仓'] }} ({{ item['目标配置'][name2 + '目标占比'] }})
                            </td>
                            <td>
                                {% if item['目标配置']['建议操作'] %}
                                    {% for op in item['目标配置']['建议操作'] %}{{ op }}<br>{% endfor %}
                                {% else %}
                                    当前配置合理，无需调整
                                {% endif %}
                            </td>
                            <td>{{ item['计算依据'] }}</td>
                        </tr>
                        {% endfor %}
                    </table>
                </div>
            </div>
        </body>
        </html>
        """
        
        # 渲染模板
        template = Template(template_str)
        html_content = template.render(
            timestamp=datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            name1=self.analyzer.name1,
            name2=self.analyzer.name2,
            shares_1=shares_1,
            shares_2=shares_2,
            cash=cash,
            value_1=value_1,
            value_2=value_2,
            total_value=total_value,
            nav_chart=nav_fig.to_html(full_html=False),
            signals_chart=signals_fig.to_html(full_html=False),
            advice=advice,
            strategy_metrics=strategy_metrics
        )
        
        # 保存报告
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        report_path = os.path.join(self.output_dir, f'hedge_strategy_report_{timestamp}.html')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        return report_path

In [ ]:
# 生成报告
report_gen = ReportGenerator(analyzer, 'output')
report_path = report_gen.generate_html_report(TEST_SHARES_1, TEST_SHARES_2, TEST_CASH)

print(f"报告已生成: {report_path}")

# 打开报告
webbrowser.open('file://' + os.path.abspath(report_path))

---
## 7. 工具函数

In [ ]:
def quick_analysis(trade_code1, trade_code2, name1, name2, shares1, shares2, cash):
    """快速对冲策略分析
    
    参数:
        trade_code1: 标的1代码
        trade_code2: 标的2代码
        name1: 标的1名称
        name2: 标的2名称
        shares1: 标的1持仓数量
        shares2: 标的2持仓数量
        cash: 可用现金
    """
    # 数据获取
    db = DatabaseManager()
    db.connect()
    
    fetcher = DataFetcher(db)
    raw_data = fetcher.get_paired_data(trade_code1, trade_code2)
    
    if raw_data.empty:
        print("数据获取失败")
        return
    
    # 数据处理
    processor = DataProcessor(raw_data, name1, name2)
    processor.process()
    df = processor.get_processed_data()
    
    # 分析
    analyzer = MarketAnalyzer(df, name1, name2)
    
    # 报告生成
    report_gen = ReportGenerator(analyzer, 'output')
    report_path = report_gen.generate_html_report(shares1, shares2, cash)
    
    # 清理
    db.disconnect()
    
    print(f"\n分析完成，报告路径: {report_path}")
    return report_path

In [ ]:
# 运行快速分析示例
quick_analysis(
    trade_code1='511220.SH',
    trade_code2='518880.SH',
    name1='城投ETF',
    name2='黄金ETF',
    shares1=10000,
    shares2=5000,
    cash=100000
)

In [ ]:
# 关闭数据库连接
db.disconnect()
print("\n分析完成！")